# 01 — TMDB Poster Scraping
Hedef: 10.000–12.000 film afişi + `labels.csv`

In [ ]:
# --- Google Colab: Drive bağlantısı ---
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/film-genre-project'

# --- Lokalde çalıştırma ---
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
print('Project root:', PROJECT_ROOT)

In [ ]:
import subprocess
import sys

SCRAPY_DIR = PROJECT_ROOT / 'src' / 'scraper'
SCRAPY_BIN = PROJECT_ROOT / '.venv' / 'Scripts' / 'scrapy.exe'

# Colab'da pip install gerekir
# !pip install scrapy requests -q

print('Scrapy dir:', SCRAPY_DIR)
print('Scrapy bin:', SCRAPY_BIN)

## 100 Filmlik Test Çekimi

In [ ]:
result = subprocess.run(
    [str(SCRAPY_BIN), 'crawl', 'tmdb',
     '-s', 'CLOSESPIDER_ITEMCOUNT=100',
     '-s', 'max_pages=10',
     '-L', 'INFO'],
    cwd=str(SCRAPY_DIR),
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
import pandas as pd

labels_path = PROJECT_ROOT / 'labels.csv'
df = pd.read_csv(labels_path)
print(f'Toplam film: {len(df)}')
df.head(10)

In [ ]:
# Sınıf dağılımını kontrol et
from collections import Counter
import matplotlib.pyplot as plt

all_genres = [g for genres in df['genres'].str.split('|') for g in genres]
genre_counts = Counter(all_genres)

genres_df = pd.DataFrame(genre_counts.most_common(), columns=['genre', 'count'])
print(genres_df.to_string())

plt.figure(figsize=(12, 5))
plt.bar(genres_df['genre'], genres_df['count'])
plt.xticks(rotation=45, ha='right')
plt.title('Tür Dağılımı (Test Çekimi)')
plt.tight_layout()
plt.show()

## Tam Çekim (10.000–12.000 Film)
Test çekimi başarılıysa aşağıdaki hücreyi çalıştır.

In [ ]:
# Önceki test verilerini temizlemek istersen:
# import shutil
# shutil.rmtree(PROJECT_ROOT / 'posters', ignore_errors=True)
# (PROJECT_ROOT / 'labels.csv').unlink(missing_ok=True)

result = subprocess.run(
    [str(SCRAPY_BIN), 'crawl', 'tmdb',
     '-L', 'WARNING'],
    cwd=str(SCRAPY_DIR),
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# Çekim sonrası özet
posters_dir = PROJECT_ROOT / 'posters'
poster_count = len(list(posters_dir.glob('*.jpg')))
df_final = pd.read_csv(labels_path)

print(f'İndirilen poster: {poster_count}')
print(f'labels.csv satır sayısı: {len(df_final)}')

all_genres = [g for genres in df_final['genres'].str.split('|') for g in genres]
genre_counts = Counter(all_genres)
print('\nTür dağılımı:')
for genre, count in sorted(genre_counts.items(), key=lambda x: -x[1]):
    status = '✓' if count >= 500 else '✗ (<500)'
    print(f'  {status} {genre}: {count}')